In [438]:
import matplotlib
matplotlib.rcParams['font.serif'] = ['Times'] + matplotlib.rcParams['font.serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [439]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
import math

import pandas as pd


import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots

In [440]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

# Helper Functions

In [441]:
#from unifying_algo_analysis_helper import *
from set_zorder_functions import *

In [442]:
def clean_labels(ax, show_labels, spines=['left', 'bottom']):
    if show_labels:
        ax.set_xticklabels(['', '$0$', '$.68$', '', '$2$', '$3$', '$4$', '$5$'])
    else:
        ax.set_xticklabels([])
        
    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    
    if show_labels:
        ax.set_yticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_yticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_ylabel('Course direction', labelpad=-2)
        ax.set_xlabel('Time relative to flash (s)')
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [443]:
TRANSLATION = True

In [444]:
FIGURE_NAME = 'unifying_analysis.svg'

In [445]:
if TRANSLATION:
    FIGURE_NAME = 'unifying_analysis_translationTrue.svg'

In [446]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import cdist

from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union
import numpy as np
from sklearn.preprocessing import MinMaxScaler

from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union

In [447]:
def _set_round_caps(ax, lines_before):
    """Set round cap/join style on any lines added since lines_before snapshot."""
    for line in ax.lines[len(lines_before):]:
        line.set_solid_capstyle('round')
        line.set_solid_joinstyle('round')

def plot_arrowhead_trajectory_scaled(x, y, color='black', arrow_length=0.05, arrow_angle=30,
                                     ax=None, linewidth=1, scale_bar=False, units='',
                                     flow_direction=None, fontsize=5,
                                     flow_arrow_length=0.05, flow_arrow_angle=30,
                                     flow_arrow_size=0.08, padding=0.2,
                                     flow_column_width=0.2,
                                     flow_arrow_label_gap=2,
                                     cmap=None, clim=None,
                                     extend_endpoint=False,
                                     scale=None,
                                     scale_bar_pos='auto'):
    """
    Parameters
    ----------
    color : str, array-like
        If a string or single color, used uniformly. If an array of the same
        length as x/y, used as scalar values mapped through `cmap`.
    cmap : str or Colormap, optional
        Colormap to use when `color` is an array of scalars. Defaults to 'viridis'.
    clim : tuple (vmin, vmax), optional
        Color limits for the colormap. Defaults to (min(color), max(color)).
    extend_endpoint: bool or int, optional
        Extend the end of the trajectory to make the arrowhead look less awkward. If not False, provide an int. The int determines
        how many steps to add at the end, through simple interpolation. Also extends color array, if color provided as an array. 
    scale_bar_pos: str, optional
        Default of auto will try to place the scale bar in an unobtrusive corner. Alternatively, supply one of the following:
        'lower_left', 'lower_right', 'upper_left', 'upper_right'
    """

    # --- Determine if we're in colormap mode ---
    color_array = np.asarray(color)
    use_cmap = cmap is not None or (color_array.ndim == 1 and color_array.dtype.kind in ('f', 'i', 'u') and len(color_array) == len(x))


    if extend_endpoint is not False:
        dx = x[-1] - x[-2]
        dy = y[-1] - y[-2]
        x = np.hstack( (x, [x[-1] + dx*i for i in range(extend_endpoint)]) )
        y = np.hstack((y, [y[-1] + dy*i for i in range(extend_endpoint)]) )
        
        if use_cmap:
            dcolor_array = color_array[-1] - color_array[-2]
            color_array  = np.hstack( (color_array, [color_array[-1] + dcolor_array*i for i in range(extend_endpoint)]) )
    
    if use_cmap:
        cmap = plt.get_cmap(cmap if cmap is not None else 'viridis')
        scalar_values = color_array.astype(float)
        vmin, vmax = clim if clim is not None else (np.nanmin(scalar_values), np.nanmax(scalar_values))
        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        # Resolve a single representative color for scale bar / flow arrow
        scalar_mean = np.nanmean(scalar_values)
        representative_color = cmap(norm(scalar_mean))
    else:
        representative_color = color

    has_flow = flow_direction is not None and flow_arrow_length is not None

    # --- Physical axis size ---
    ax.figure.canvas.draw()
    bbox = ax.get_window_extent().transformed(ax.figure.dpi_scale_trans.inverted())
    ax_width_in  = bbox.width
    ax_height_in = bbox.height
    coord_width  = ax_width_in
    coord_height = ax_height_in

    flow_col = flow_column_width * coord_width if has_flow else 0.0

    left_x_min = padding * coord_width
    left_x_max = coord_width - flow_col - padding * coord_width
    y_min_pad  = padding * coord_height
    y_max_pad  = coord_height - padding * coord_height

    x_range = np.nanmax(x) - np.nanmin(x)
    y_range = np.nanmax(y) - np.nanmin(y)

    available_w = left_x_max - left_x_min
    available_h = y_max_pad - y_min_pad

    # Use provided scale or compute it, then center the data
    if scale is None:
        scale = min(available_w / x_range, available_h / y_range)
    else:
        max_scale = min(available_w / x_range, available_h / y_range)
        if scale > max_scale:
            import warnings
            warnings.warn(
                f"Provided scale ({scale:.4f}) exceeds the maximum fitting scale "
                f"({max_scale:.4f}) for this trajectory. Data will overflow the padded region."
            )

    x_norm = (x - np.nanmin(x)) * scale
    y_norm = (y - np.nanmin(y)) * scale

    x_offset = left_x_min + (available_w - x_range * scale) / 2
    y_offset = y_min_pad  + (available_h - y_range * scale) / 2

    x_scaled = x_norm + x_offset
    y_scaled = y_norm + y_offset

    ax.set_xlim(0, coord_width)
    ax.set_ylim(0, coord_height)
    ax.set_aspect('equal')

    # --- Plot trajectory: segment-by-segment if using colormap ---
    if use_cmap:
        # Plot each segment individually with its mapped color.
        # The arrowhead is drawn only on the final segment so it inherits the
        # terminal color — pass a dummy no-arrow call for all-but-last segments.
        n = len(x_scaled)
        for i in range(n - 2):
            seg_color = cmap(norm(scalar_values[i]))
            lines_before = list(ax.lines)
            # Only draw the arrowhead decoration on the last segment
            braid_analysis_plots.plot_arrowhead_trajectory(
                x_scaled[i:i+3], y_scaled[i:i+3],
                color=seg_color,
                arrow_length=arrow_length if i == n - 3 else 0,
                arrow_angle=arrow_angle,
                ax=ax,
                linewidth=linewidth
            )
            _set_round_caps(ax, lines_before)
    else:
        braid_analysis_plots.plot_arrowhead_trajectory(
            x_scaled, y_scaled,
            color=color,
            arrow_length=arrow_length,
            arrow_angle=arrow_angle,
            ax=ax,
            linewidth=linewidth
        )

    for collection in ax.collections:
        collection.set_clip_on(False)

    # --- Reread limits ---
    ax.figure.canvas.draw()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    x_extent_ax = xlim[1] - xlim[0]
    y_extent_ax = ylim[1] - ylim[0]

    label_gap = 0.03 * y_extent_ax
    offset_x  = 0.02 * x_extent_ax

    right_x_min = coord_width - flow_col
    right_x_max = coord_width

    ax.set_aspect('equal')

    traj_hull = MultiPoint(list(zip(x_scaled, y_scaled))).convex_hull.buffer(offset_x * 2)
    occupied_geom = traj_hull

    def find_best_position(size, occupied, x_min, x_max, y_min, y_max):
        n = 30
        xs = np.linspace(x_min + size, x_max - size, n)
        ys = np.linspace(y_min + size, y_max - size, n)
        best_pt, best_dist = None, -1
        for cx_ in xs:
            for cy_ in ys:
                pt = Point(cx_, cy_)
                d = pt.distance(occupied) if not occupied.is_empty else 1e9
                box_fits = (cx_ - size > x_min and cx_ + size < x_max and
                            cy_ - size > y_min and cy_ + size < y_max)
                if box_fits and d > best_dist:
                    best_dist = d
                    best_pt = (cx_, cy_)
        return best_pt

    # --- Scale bar (uses representative_color) ---
    if scale_bar:
        x_extent_original = np.nanmax(x) - np.nanmin(x)
        max_bar_original = 0.3 * x_extent_original
        magnitude = 10 ** np.floor(np.log10(max_bar_original))
        nice_steps = [1, 2, 5]
        bar_size_original = magnitude
        for step in nice_steps:
            candidate = step * magnitude
            if candidate <= max_bar_original:
                bar_size_original = candidate

        scale_factor = (left_x_max - left_x_min) / x_extent_original
        bar_size_scaled = bar_size_original * scale_factor
        bar_half = bar_size_scaled / 2

        #pos = find_best_position(max(bar_half, label_gap * 2), occupied_geom,
        #                         xlim[0], right_x_min, ylim[0], ylim[1])
        if scale_bar_pos == 'auto':
            pos = find_best_position(max(bar_half, label_gap * 2), occupied_geom,
                                     xlim[0], right_x_min, ylim[0], ylim[1])
        else:
            inset_x = bar_half + offset_x
            inset_y = label_gap * 3

            positions = {
                'lower_left':  (xlim[0]    + inset_x, ylim[0] + inset_y),
                'lower_right': (right_x_min - inset_x, ylim[0] + inset_y),
                'upper_left':  (xlim[0]    + inset_x, ylim[1] - inset_y),
                'upper_right': (right_x_min - inset_x, ylim[1] - inset_y),
            }

            if scale_bar_pos not in positions:
                raise ValueError(
                    f"scale_bar_pos must be 'auto', 'lower_left', 'lower_right', "
                    f"'upper_left', or 'upper_right'. Got '{scale_bar_pos}'."
                )

            pos = positions[scale_bar_pos]
        
        if pos is not None:
            cx, cy = pos
            bar_x_start = cx - bar_half
            bar_x_end   = cx + bar_half

            if cy < (ylim[0] + ylim[1]) / 2:
                text_y, va = cy + label_gap, 'bottom'
            else:
                text_y, va = cy - label_gap, 'top'

            ax.plot([bar_x_start, bar_x_end], [cy, cy],
                    color=representative_color, linewidth=0.5)
            ax.text(cx, text_y, f'{bar_size_original:g} {units}',
                    ha='center', va=va, fontsize=fontsize, color=representative_color)

            bar_geom = MultiPoint([
                (bar_x_start, cy), (bar_x_end, cy), (cx, text_y)
            ]).convex_hull.buffer(label_gap * 2)
            occupied_geom = unary_union([occupied_geom, bar_geom])

    # --- Flow direction arrow (uses representative_color) ---
    if has_flow:
        arrow_size_scaled = flow_arrow_size * x_extent_ax

        cx = (right_x_min + right_x_max) / 2
        cy = (ylim[0] + ylim[1]) / 2

        dx = np.cos(flow_direction) * arrow_size_scaled / 2
        dy = np.sin(flow_direction) * arrow_size_scaled / 2

        n_points = 10
        t = np.linspace(-0.5, 0.5, n_points)
        arrow_x = cx + t * dx * 2
        arrow_y = cy + t * dy * 2

        braid_analysis_plots.plot_arrowhead_trajectory(
            arrow_x, arrow_y,
            color=representative_color,
            arrow_length=flow_arrow_length,
            arrow_angle=flow_arrow_angle,
            ax=ax,
            linewidth=linewidth
        )

        for collection in ax.collections:
            collection.set_clip_on(False)

        text_angle_deg = np.degrees(flow_direction)
        if 90 < text_angle_deg % 360 < 270:
            text_angle_deg += 180

        perp_dx = -np.sin(flow_direction) * label_gap * flow_arrow_label_gap
        perp_dy =  np.cos(flow_direction) * label_gap * flow_arrow_label_gap
        cx_mid = (right_x_min + right_x_max) / 2
        cy_mid = (ylim[0] + ylim[1]) / 2
        if (cx + perp_dx - cx_mid)**2 + (cy + perp_dy - cy_mid)**2 < \
           (cx - perp_dx - cx_mid)**2 + (cy - perp_dy - cy_mid)**2:
            perp_dx, perp_dy = -perp_dx, -perp_dy

        ax.text(cx + perp_dx, cy + perp_dy, 'flow',
                ha='center', va='center', fontsize=fontsize, color=representative_color,
                rotation=text_angle_deg, rotation_mode='anchor')



In [448]:
def make_gray_red_black_cmap():
    """
    Custom colormap:
      - values < 0  → gray
      - value  = 0  → transparent ('none')
      - 0 < value < 1 → black
      - value  = 1  → red
      - values > 1  → clipped to red
    """
    colors = [
        (0.0,    'gray'),
        (0.4999, 'gray'),
        (0.5,    (0, 0, 0, 0)),  # transparent at value=0
        (0.5001, 'black'),
        (0.9999, 'black'),
        (1.0,    'red'),
    ]
    cmap = LinearSegmentedColormap.from_list('gray_red_black', colors)
    return cmap

def make_gray_red_black_timeseries(trajec):
    color = np.sign(trajec.time_relative_to_flash.values) + trajec.lights_on.values / 100
    return color

# With real laminar data

In [488]:
show_labels = False

In [489]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

/home/caveman/PY38/lib/python3.8/site-packages/figurefirst/svg_to_axes.py:1041: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(fw_in, fh_in))


In [490]:
braid_df_fname =  '../trajectory_data/flies_laminar_c1xwt_preprocessed_optotrigger_trimmed.hdf'
df_laminar = pd.read_hdf(braid_df_fname)

In [491]:
# get a single trajectory
braid_df = df_laminar[df_laminar.intensity==100]
obj_id_key = 'obj_id_unique_event'
obj_id = braid_df[obj_id_key].unique()[40] # << 36 is a good demo, 40 is beautiful, 41 is nice
trajec = braid_df[braid_df[obj_id_key]==obj_id]
trajec = trajec.dropna()

In [492]:
if 0:
    braid_analysis_plots.plot_arrowhead_trajectory(trajec.x.values, trajec.y.values)
else:
    ax = layout.axes[('laminar_example', 'trajec')]

    x_pos = trajec.x.values
    y_pos = trajec.y.values
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[0:20], y_pos[0:20], color='gray', linewidth=1, ax=ax, arrow_length=0)
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[20:87], y_pos[20:87], color='red', linewidth=1, ax=ax, arrow_length=0)
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[87:], y_pos[87:], color='black', linewidth=1, ax=ax, arrow_length=0.03)
    
    
    #ax.set_xlim(-0.15, 0.15)
    #ax.set_ylim(-0.15, 0.05)
    #ax.set_aspect('equal')

    grb_cmap = make_gray_red_black_cmap()
    color = make_gray_red_black_timeseries(trajec)
    plot_arrowhead_trajectory_scaled(x_pos, y_pos, color=color, arrow_length=0.1, arrow_angle=30,
                                     ax=ax, linewidth=1, scale_bar=True, units='m',
                                     flow_direction=None, fontsize=5,
                                     flow_arrow_length=0.05, flow_arrow_angle=30,
                                     flow_arrow_size=0.08, padding=0.05,
                                     flow_column_width=0.2,
                                     flow_arrow_label_gap=2,
                                     cmap=grb_cmap, clim=None,
                                     extend_endpoint=30,
                                     scale=2.5)
    
    fifi.mpl_functions.adjust_spines(ax, [])

set_selective_rasterization(ax, rasterize_collections=[mcollections.PatchCollection], raster_zorder=-2)
set_selective_rasterization(ax, rasterize_linestyles=['-'], raster_zorder=-2)


In [493]:
course = trajec.course_smoothish.values
abs_min_ix = 120 # 1 second after the flash start; roughly 300 ms after flash end. Adjust based on your data.
abs_max_ix = 500 # 5 seconds after the flash start. 
min_ix_range = 190 # Use a ~2 sec window for fit
max_ix_range = 210 # Use a ~2 sec window for fit

In [494]:
unifying_algo_fit = uaa.bootstrap_miop_affine_fit(course, 
                                                  abs_min_ix, abs_max_ix, min_ix_range, max_ix_range, 
                                                  npoints = 50, 
                                                  n_bootstraps=10, 
                                                  use_cvx_affine=True, 
                                                  include_translation=TRANSLATION)

In [495]:
# Sort 
sorted_fit = unifying_algo_fit.sort_values('rmse_affine')
best_ix = sorted_fit.index[0]
best_unifying_algo_fit = unifying_algo_fit.iloc[best_ix]

In [496]:
sorted_fit.keys()

Index(['min_ix', 'max_ix', 'abs_min_ix', 'abs_max_ix', 'min_ix_range',
       'max_ix_range', 'slope', 'intercept', 'rotation', 'major_axis',
       'minor_axis', 'AM_0_0', 'AM_0_1', 'AM_1_0', 'AM_1_1', 'tx', 'ty', 'rho',
       'mean_residuals', 'axis_ratio', 'rmse_affine', 'shift_ix', 'iteration',
       'obj_id_unique_event', 'timestep_sec'],
      dtype='object')

In [497]:
ax = layout.axes[('laminar_example', 'course')]
uap.plot_model_fit(course, best_unifying_algo_fit, 
                   flash_frame_start=20,
                   flash_frame_end=88,
                   clean_spines=True,
                   ax=ax,
                   course_marker_size=2,
                   linear_marker_size=2,
                   affine_marker_size=2,
                  )

In [498]:
print('median slope: ', unifying_algo_fit['slope'].abs().median())
print('median axis ratio: ', unifying_algo_fit['axis_ratio'].abs().median())
print('weighted median: ', uaa.get_weighted_value(unifying_algo_fit, col='axis_ratio', use='rmse_affine'))

median slope:  0.046229281527977756
median axis ratio:  0.10748700466569264
weighted median:  0.12544559548567769


In [499]:
clean_labels(ax, show_labels)

In [500]:
set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)

layout.append_figure_to_layer(layout.figures['laminar_example'], 'laminar_example', cleartarget=True)
layout.write_svg(FIGURE_NAME)

# With real still air data

In [462]:
show_labels = True

In [463]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

/home/caveman/PY38/lib/python3.8/site-packages/figurefirst/svg_to_axes.py:1041: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(fw_in, fh_in))


In [464]:
braid_df_fname =  '../trajectory_data/flies_stillair_c1xwt_preprocessed_optotrigger_trimmed.hdf'
df_still = pd.read_hdf(braid_df_fname)

In [465]:
# get a single trajectory
braid_df = df_still[df_still.intensity==100]
obj_id_key = 'obj_id_unique_event'
# obj_id = braid_df[obj_id_key].unique()[36] << 36 is a good demo, 29 & 35 is a good one that switches direction of circling
obj_id = braid_df[obj_id_key].unique()[36] # '20220817_173045_4200_104' is a tricky one
trajec = braid_df[braid_df[obj_id_key]=='20220817_173045_4200_104']
trajec = trajec.dropna()

In [466]:
if 0:
    braid_analysis_plots.plot_arrowhead_trajectory(trajec.x.values, trajec.y.values)
else:
    ax = layout.axes[('stillair_example', 'trajec')]

    x_pos = trajec.x.values
    y_pos = trajec.y.values
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[0:20], y_pos[0:20], color='gray', linewidth=1, ax=ax, arrow_length=0)
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[20:87], y_pos[20:87], color='red', linewidth=1, ax=ax, arrow_length=0)
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[87:], y_pos[87:], color='black', linewidth=1, ax=ax, arrow_length=0.03)
    
    
    #ax.set_xlim(-0.15, 0.15)
    #ax.set_ylim(-0.15, 0.05)
    #ax.set_aspect('equal')

    grb_cmap = make_gray_red_black_cmap()
    color = make_gray_red_black_timeseries(trajec)
    plot_arrowhead_trajectory_scaled(x_pos, y_pos, color=color, arrow_length=0.1, arrow_angle=30,
                                     ax=ax, linewidth=1, scale_bar=True, units='m',
                                     flow_direction=None, fontsize=5,
                                     flow_arrow_length=0.05, flow_arrow_angle=30,
                                     flow_arrow_size=0.08, padding=0.05,
                                     flow_column_width=0.2,
                                     flow_arrow_label_gap=2,
                                     cmap=grb_cmap, clim=None,
                                     extend_endpoint=30,
                                     scale=1.5,
                                     scale_bar_pos='lower_left')
    
    fifi.mpl_functions.adjust_spines(ax, [])

set_selective_rasterization(ax, rasterize_collections=[mcollections.PatchCollection], raster_zorder=-2)
set_selective_rasterization(ax, rasterize_linestyles=['-'], raster_zorder=-2)


In [467]:
course = trajec.course_smoothish.values
abs_min_ix = 120 # 1 second after the flash start; roughly 300 ms after flash end. Adjust based on your data.
abs_max_ix = 500 # 5 seconds after the flash start. 
min_ix_range = 190 # Use a ~2 sec window for fit
max_ix_range = 210 # Use a ~2 sec window for fit

In [468]:
unifying_algo_fit = uaa.bootstrap_miop_affine_fit(course, 
                                                  abs_min_ix, abs_max_ix, min_ix_range, max_ix_range, 
                                                  npoints = 50, 
                                                  n_bootstraps=10, 
                                                  use_cvx_affine=True, 
                                                  include_translation=TRANSLATION)

In [469]:
# Sort 
sorted_fit = unifying_algo_fit.sort_values('rmse_affine')
best_ix = sorted_fit.index[0]
best_unifying_algo_fit = unifying_algo_fit.iloc[best_ix]

In [470]:
sorted_fit.keys()

Index(['min_ix', 'max_ix', 'abs_min_ix', 'abs_max_ix', 'min_ix_range',
       'max_ix_range', 'slope', 'intercept', 'rotation', 'major_axis',
       'minor_axis', 'AM_0_0', 'AM_0_1', 'AM_1_0', 'AM_1_1', 'tx', 'ty', 'rho',
       'mean_residuals', 'axis_ratio', 'rmse_affine', 'shift_ix', 'iteration',
       'obj_id_unique_event', 'timestep_sec'],
      dtype='object')

In [471]:
ax = layout.axes[('stillair_example', 'course')]
uap.plot_model_fit(course, best_unifying_algo_fit, 
                   flash_frame_start=20,
                   flash_frame_end=88,
                   clean_spines=True,
                   ax=ax,
                   course_marker_size=2,
                   linear_marker_size=2,
                   affine_marker_size=2,
                  )

In [472]:
print('median slope: ', unifying_algo_fit['slope'].abs().median())
print('median axis ratio: ', unifying_algo_fit['axis_ratio'].abs().median())
print('weighted median: ', uaa.get_weighted_value(unifying_algo_fit, col='axis_ratio', use='rmse_affine'))

median slope:  0.03104043483799404
median axis ratio:  0.3557282593484212
weighted median:  0.3274244268357213


In [473]:
clean_labels(ax, show_labels)

In [474]:
set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)

layout.append_figure_to_layer(layout.figures['stillair_example'], 'stillair_example', cleartarget=True)
layout.write_svg(FIGURE_NAME)

# With real unsteady data

In [475]:
show_labels = False

In [476]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

/home/caveman/PY38/lib/python3.8/site-packages/figurefirst/svg_to_axes.py:1041: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(fw_in, fh_in))


In [477]:
braid_df_fname =  '../trajectory_data/flies_splitflow_topon_c1xwt_preprocessed_optotrigger_trimmed.hdf'
df_unsteady = pd.read_hdf(braid_df_fname)

In [478]:
# get a single trajectory
braid_df = df_unsteady[df_unsteady.intensity>0]
obj_id_key = 'obj_id_unique_event'
obj_id = braid_df[obj_id_key].unique()[2] # 0
trajec = braid_df[braid_df[obj_id_key]==obj_id]
trajec = trajec.dropna()

In [479]:
if 0:
    braid_analysis_plots.plot_arrowhead_trajectory(trajec.x.values, trajec.y.values)
else:
    ax = layout.axes[('unsteady_example', 'trajec')]

    x_pos = trajec.x.values
    y_pos = trajec.y.values
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[0:20], y_pos[0:20], color='gray', linewidth=1, ax=ax, arrow_length=0)
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[20:87], y_pos[20:87], color='red', linewidth=1, ax=ax, arrow_length=0)
    #braid_analysis_plots.plot_arrowhead_trajectory(x_pos[87:], y_pos[87:], color='black', linewidth=1, ax=ax, arrow_length=0.03)
    
    
    #ax.set_xlim(-0.15, 0.15)
    #ax.set_ylim(-0.15, 0.05)
    #ax.set_aspect('equal')

    grb_cmap = make_gray_red_black_cmap()
    color = make_gray_red_black_timeseries(trajec)
    plot_arrowhead_trajectory_scaled(x_pos, y_pos, color=color, arrow_length=0.1, arrow_angle=30,
                                     ax=ax, linewidth=1, scale_bar=True, units='m',
                                     flow_direction=None, fontsize=5,
                                     flow_arrow_length=0.05, flow_arrow_angle=30,
                                     flow_arrow_size=0.08, padding=0.05,
                                     flow_column_width=0.2,
                                     flow_arrow_label_gap=2,
                                     cmap=grb_cmap, clim=None,
                                     extend_endpoint=30,
                                     scale=1.5)
    
    fifi.mpl_functions.adjust_spines(ax, [])

set_selective_rasterization(ax, rasterize_collections=[mcollections.PatchCollection], raster_zorder=-2)
set_selective_rasterization(ax, rasterize_linestyles=['-'], raster_zorder=-2)


In [480]:
course = trajec.course_smoothish.values
abs_min_ix = 120 # 1 second after the flash start; roughly 300 ms after flash end. Adjust based on your data.
abs_max_ix = 500 # 5 seconds after the flash start. 
min_ix_range = 190 # Use a ~2 sec window for fit
max_ix_range = 210 # Use a ~2 sec window for fit

In [481]:
unifying_algo_fit = uaa.bootstrap_miop_affine_fit(course, 
                                                  abs_min_ix, abs_max_ix, min_ix_range, max_ix_range, 
                                                  npoints = 50, 
                                                  n_bootstraps=10, 
                                                  use_cvx_affine=True, 
                                                  include_translation=TRANSLATION)

In [482]:
# Sort 
sorted_fit = unifying_algo_fit.sort_values('rmse_affine')
best_ix = sorted_fit.index[0]
best_unifying_algo_fit = unifying_algo_fit.iloc[best_ix]

In [483]:
sorted_fit.keys()

Index(['min_ix', 'max_ix', 'abs_min_ix', 'abs_max_ix', 'min_ix_range',
       'max_ix_range', 'slope', 'intercept', 'rotation', 'major_axis',
       'minor_axis', 'AM_0_0', 'AM_0_1', 'AM_1_0', 'AM_1_1', 'tx', 'ty', 'rho',
       'mean_residuals', 'axis_ratio', 'rmse_affine', 'shift_ix', 'iteration',
       'obj_id_unique_event', 'timestep_sec'],
      dtype='object')

In [484]:
ax = layout.axes[('unsteady_example', 'course')]
uap.plot_model_fit(course, best_unifying_algo_fit, 
                   flash_frame_start=20,
                   flash_frame_end=88,
                   clean_spines=True,
                   ax=ax,
                   course_marker_size=2,
                   linear_marker_size=2,
                   affine_marker_size=2,
                  )

In [485]:
print('median slope: ', unifying_algo_fit['slope'].abs().median())
print('median axis ratio: ', unifying_algo_fit['axis_ratio'].abs().median())
print('weighted median: ', uaa.get_weighted_value(unifying_algo_fit, col='axis_ratio', use='rmse_affine'))

median slope:  0.05324511061386335
median axis ratio:  0.18882812106097008
weighted median:  0.17193693085189524


In [486]:
clean_labels(ax, show_labels)

In [487]:
set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)

layout.append_figure_to_layer(layout.figures['unsteady_example'], 'unsteady_example', cleartarget=True)
layout.write_svg(FIGURE_NAME)

In [48]:
data = diagnose_axis_elements(ax)

COLLECTIONS (scatter, fill_between, etc.)

Collection 0:
  Type: PolyCollection
  Full type: <class 'matplotlib.collections.PolyCollection'>
  Label: _child0
  Current zorder: -2

Collection 1:
  Type: PolyCollection
  Full type: <class 'matplotlib.collections.PolyCollection'>
  Label: _child1
  Current zorder: -2

LINES (plot, with markers and linestyles)

Line 0:
  Label: _child2
  Marker: '.'
  Linestyle: 'None'
  Color: black
  Linewidth: 1.5
  Current zorder: -1

Line 1:
  Label: _child3
  Marker: '.'
  Linestyle: 'None'
  Color: blue
  Linewidth: 1.5
  Current zorder: -1

Line 2:
  Label: _child4
  Marker: '.'
  Linestyle: 'None'
  Color: magenta
  Linewidth: 1.5
  Current zorder: -1

SUMMARY

Total collections: 2

Collection types found:
  PolyCollection: 2

Total lines: 3

Marker types found:
  '.': 3

RASTERIZATION SUGGESTIONS

To rasterize collections, use:
  rasterize_collections=[
      mcollections.PolyCollection,
  ]

To rasterize by marker type, use:
  rasterize_markers=